# Spam and Malware Detection on Google Colab

This notebook loads the project ZIP, installs its dependencies, checks the project files, and launches the Flask app through Colab's port proxy. Sign in to Google Colab before running it. The ZIP excludes local credentials and cached datasets; the trained spam and malware model artifacts are included.

## 1. Mount Google Drive

Mount Drive if you want to select the archive from `MyDrive`. You can also skip this cell and upload the ZIP directly in the next section.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Upload Project Files to Colab

Run the cell and select `spam-malware-detection-project.zip` from your computer. Alternatively, place that ZIP in the root of Google Drive and skip the upload cell; the code checks that location as a fallback.

In [ ]:
from google.colab import files
from pathlib import Path

project_zip = Path('/content/drive/MyDrive/spam-malware-detection-project.zip')
if project_zip.is_file():
    print(f'Using Drive archive: {project_zip}')
else:
    uploaded = files.upload()
    zip_candidates = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
    if not zip_candidates:
        raise FileNotFoundError('Upload spam-malware-detection-project.zip or place it in the root of MyDrive.')
    project_zip = zip_candidates[0]
print(f'Project archive: {project_zip} ({project_zip.stat().st_size / (1024 * 1024):.2f} MiB)')

## 3. Unzip or Extract Project Files

The extraction cell checks archive paths before writing files, then uses the ZIP's project folder under `/content`.

In [ ]:
import zipfile

content_root = Path('/content').resolve()
with zipfile.ZipFile(project_zip) as archive:
    for member in archive.infolist():
        destination = (content_root / member.filename).resolve()
        if destination != content_root and content_root not in destination.parents:
            raise ValueError(f'Unsafe archive path: {member.filename}')
    archive.extractall(content_root)

project_dir = content_root / 'spam-malware-detection-project'
if not (project_dir / 'app.py').is_file():
    raise FileNotFoundError(f'app.py was not found under {project_dir}')
print(f'Extracted project to {project_dir}')

## 4. Verify Project Structure

Confirm the source, both trained model artifacts, and requirements file are present before installing or starting anything.

In [ ]:
required_files = [
    'app.py',
    'requirements.txt',
    'models/spam_model.joblib',
    'models/malware_model.joblib',
    'models/malware_scaler.joblib',
]
missing_files = [name for name in required_files if not (project_dir / name).is_file()]
if missing_files:
    raise FileNotFoundError(f'Missing project files: {missing_files}')
for path in sorted(project_dir.rglob('*')):
    if path.is_file():
        print(path.relative_to(project_dir))

## 5. Install Project Dependencies

Install the packages pinned by the project's `requirements.txt` into the active Colab runtime.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(project_dir / 'requirements.txt')],
    check=True,
)

## 6. Run Project Code in Colab

Use the bundled trained models when available. If either model is missing, the cell trains it. Then Flask starts in the background and Colab exposes port 5000. On a first run, open the app's `/configure-login` page to create a local account.

In [ ]:
import os
import subprocess
import sys
import time
import urllib.error
import urllib.request
from IPython.display import Markdown, display

spam_model_path = project_dir / 'models' / 'spam_model.joblib'
malware_model_path = project_dir / 'models' / 'malware_model.joblib'
malware_scaler_path = project_dir / 'models' / 'malware_scaler.joblib'
if not spam_model_path.is_file():
    subprocess.run([sys.executable, 'train_spam_model.py'], cwd=project_dir, check=True)
if not malware_model_path.is_file() or not malware_scaler_path.is_file():
    subprocess.run([sys.executable, 'train_malware_model.py'], cwd=project_dir, check=True)

flask_log = open('/content/flask_app.log', 'w', encoding='utf-8')
flask_environment = os.environ.copy()
flask_environment.setdefault('PYTHONUNBUFFERED', '1')
flask_process = subprocess.Popen(
    [sys.executable, 'app.py'],
    cwd=project_dir,
    env=flask_environment,
    stdout=flask_log,
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    if flask_process.poll() is not None:
        flask_log.flush()
        raise RuntimeError(Path('/content/flask_app.log').read_text(encoding='utf-8'))
    try:
        with urllib.request.urlopen('http://127.0.0.1:5000/login', timeout=2):
            break
    except (urllib.error.URLError, TimeoutError):
        time.sleep(1)
else:
    raise TimeoutError('Flask did not start on port 5000. See /content/flask_app.log.')

from google.colab import output
app_url = output.eval_js('google.colab.kernel.proxyPort(5000)')
display(Markdown(f'**Flask app is running:** [Open the project]({app_url})'))
print('If this is your first run, open /configure-login from the app and create an account.')